# 🛡️ GigShield — FT-Transformer Premium Pricing Model

**Model:** Feature Tokenizer Transformer (Gorishniy et al. 2021)
**Purpose:** Predict personalized weekly premium (₹29–₹99) from 7 rider features
**Output:** `premium_paise` + `attention_weights` (XAI factors)

This notebook:
1. Generates 50,000 synthetic rider profiles
2. Trains the FT-Transformer model
3. Saves model weights + normalization params
4. Tests inference with sample riders

In [ ]:
# Install dependencies (run in Colab)
!pip install torch torchvision numpy pandas scikit-learn -q

## Step 1: Generate 50,000 Synthetic Rider Profiles

In [ ]:
import numpy as np
import pandas as pd

# Zone risk profiles (from zones.json)
ZONES = {
    'BTM_LAYOUT': {'risk': 0.87, 'aqi_exp': 0.42, 'weight': 0.15},
    'KORAMANGALA': {'risk': 0.82, 'aqi_exp': 0.35, 'weight': 0.18},
    'INDIRANAGAR': {'risk': 0.91, 'aqi_exp': 0.48, 'weight': 0.15},
    'WHITEFIELD': {'risk': 0.79, 'aqi_exp': 0.30, 'weight': 0.10},
    'JAYANAGAR': {'risk': 0.85, 'aqi_exp': 0.38, 'weight': 0.08},
    'MARATHAHALLI': {'risk': 0.76, 'aqi_exp': 0.25, 'weight': 0.08},
    'HSR_LAYOUT': {'risk': 0.88, 'aqi_exp': 0.45, 'weight': 0.10},
    'ELECTRONIC_CITY': {'risk': 0.72, 'aqi_exp': 0.20, 'weight': 0.05},
    'HEBBAL': {'risk': 0.80, 'aqi_exp': 0.32, 'weight': 0.06},
    'JP_NAGAR': {'risk': 0.83, 'aqi_exp': 0.36, 'weight': 0.05},
}

def generate_synthetic_riders(n_samples=50000, seed=42):
    np.random.seed(seed)
    zone_ids = list(ZONES.keys())
    zone_weights = [ZONES[z]['weight'] for z in zone_ids]
    zones_sampled = np.random.choice(zone_ids, size=n_samples, p=zone_weights)

    data = {
        'zone_id': zones_sampled,
        'zone_risk_score': [ZONES[z]['risk'] for z in zones_sampled],
        'aqi_exposure_score': np.clip(np.random.beta(2, 5, n_samples), 0, 1),
        'work_hours_per_day': np.clip(np.random.normal(12, 2, n_samples), 6, 16).astype(int),
        'work_days_per_week': np.random.choice([5, 6, 7], n_samples, p=[0.2, 0.5, 0.3]),
        'season_multiplier': np.random.choice([0.90, 1.00, 1.15], n_samples, p=[0.25, 0.35, 0.40]),
        'claim_history_count': np.clip(np.random.poisson(0.3, n_samples), 0, 10),
        'daily_earning_bucket': np.random.choice([0, 1, 2, 3, 4], n_samples, p=[0.05, 0.25, 0.45, 0.20, 0.05]),
    }
    df = pd.DataFrame(data)

    # Target premium formula
    base = 5000  # ₹50 base in paise
    df['target_premium_paise'] = (
        base * (
            0.30 * df['zone_risk_score'] +
            0.25 * df['aqi_exposure_score'] +
            0.20 * df['season_multiplier'] +
            0.15 * (df['work_hours_per_day'] / 16) +
            0.10 * (df['work_days_per_week'] / 7)
        )
    ).astype(int)

    # Add noise (±10%)
    noise = np.random.normal(0, 300, n_samples).astype(int)
    df['target_premium_paise'] = np.clip(df['target_premium_paise'] + noise, 2900, 9900)
    return df

df = generate_synthetic_riders()
print(f'Generated {len(df)} rider profiles')
print(f'Premium stats: mean={df["target_premium_paise"].mean():.0f}, min={df["target_premium_paise"].min()}, max={df["target_premium_paise"].max()}')
df.head()

## Step 2: Define FT-Transformer Model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class FTTransformer(nn.Module):
    """
    Feature Tokenizer Transformer for premium pricing.
    Architecture:
    1. FeatureTokenizer: Projects each feature to d_token-dimensional embedding
    2. [CLS] token prepended
    3. Transformer encoder (multi-head self-attention)
    4. [CLS] output -> MLP -> premium prediction
    The attention weights from the last layer are the XAI factors.
    """
    def __init__(self, n_features=7, d_token=64, n_heads=4, n_layers=3, d_ffn=128, dropout=0.1):
        super().__init__()
        self.n_features = n_features
        self.d_token = d_token
        self.feature_tokenizers = nn.ModuleList([nn.Linear(1, d_token) for _ in range(n_features)])
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_token))
        self.pos_embedding = nn.Parameter(torch.randn(1, n_features + 1, d_token))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_token, nhead=n_heads, dim_feedforward=d_ffn,
            dropout=dropout, batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.output_head = nn.Sequential(
            nn.LayerNorm(d_token),
            nn.Linear(d_token, d_ffn),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ffn, 1),
        )
        self._last_attention_weights = None

    def forward(self, x):
        batch_size = x.size(0)
        tokens = []
        for i, tokenizer in enumerate(self.feature_tokenizers):
            feature_token = tokenizer(x[:, i:i+1])
            tokens.append(feature_token.unsqueeze(1))
        tokens = torch.cat(tokens, dim=1)
        cls = self.cls_token.expand(batch_size, -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)
        tokens = tokens + self.pos_embedding
        encoded = self.transformer(tokens)
        cls_output = encoded[:, 0, :]
        premium = self.output_head(cls_output)
        feature_outputs = encoded[:, 1:, :]
        cls_expanded = cls_output.unsqueeze(1).expand_as(feature_outputs)
        similarities = (cls_expanded * feature_outputs).sum(dim=-1)
        attention_weights = F.softmax(similarities / math.sqrt(self.d_token), dim=-1)
        self._last_attention_weights = attention_weights.detach()
        return premium, attention_weights

print('FT-Transformer model defined ✅')

## Step 3: Train the Model

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

FEATURE_COLS = ['zone_risk_score', 'aqi_exposure_score', 'work_hours_per_day',
                'work_days_per_week', 'season_multiplier', 'claim_history_count',
                'daily_earning_bucket']

# Prepare tensors
X = df[FEATURE_COLS].values.astype(np.float32)
y = df['target_premium_paise'].values.astype(np.float32).reshape(-1, 1)

# Normalize features
X_mean = X.mean(axis=0)
X_std = X.std(axis=0) + 1e-8
X_norm = (X - X_mean) / X_std

# Split 80/20
n_train = int(0.8 * len(X_norm))
X_train, X_val = X_norm[:n_train], X_norm[n_train:]
y_train, y_val = y[:n_train], y[n_train:]

train_ds = TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train))
val_ds = TensorDataset(torch.FloatTensor(X_val), torch.FloatTensor(y_val))
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {device}')

model = FTTransformer(n_features=7).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
criterion = nn.MSELoss()

best_val_loss = float('inf')
for epoch in range(50):
    model.train()
    train_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        pred, _ = model(X_batch)
        loss = criterion(pred, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            pred, _ = model(X_batch)
            val_loss += criterion(pred, y_batch).item()
    val_loss /= len(val_loader)

    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1:3d} | Train Loss: {train_loss:10.2f} | Val Loss: {val_loss:10.2f}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'premium_model_weights.pt')

print(f'\n✅ Best validation loss: {best_val_loss:.2f}')
print('✅ Model saved to premium_model_weights.pt')

## Step 4: Save Normalization Params

In [ ]:
np.savez('premium_norm_params.npz', mean=X_mean, std=X_std)
print('✅ Normalization params saved to premium_norm_params.npz')
print(f'Feature means: {X_mean}')
print(f'Feature stds: {X_std}')

## Step 5: Test Inference

In [ ]:
# Load best model
model.load_state_dict(torch.load('premium_model_weights.pt', map_location=device))
model.eval()

# Test with sample rider: Ravi from BTM Layout
test_features = np.array([[0.87, 0.42, 12.0, 6.0, 1.15, 0.0, 2.0]], dtype=np.float32)
test_norm = (test_features - X_mean) / X_std
test_tensor = torch.FloatTensor(test_norm).to(device)

with torch.no_grad():
    premium_pred, attn_weights = model(test_tensor)

premium = max(2900, min(9900, int(premium_pred.item())))
weights = attn_weights[0].cpu().numpy()

XAI_LABELS = ['zone_risk', 'aqi_exposure', 'work_hours', 'work_days', 'season', 'claim_history', 'earning_bucket']

print(f'\n🛡️ Premium prediction for demo rider:')
print(f'   Premium: ₹{premium/100:.0f}/week ({premium} paise)')
print(f'\n📊 XAI Attention Weights:')
for label, w in zip(XAI_LABELS, weights):
    bar = '█' * int(w * 50)
    print(f'   {label:20s} {w:.3f} {bar}')

## Step 6: Download Model Files

Download these files and place them in `backend/ml/premium/`:
- `premium_model_weights.pt` → rename to `model_weights.pt`
- `premium_norm_params.npz` → rename to `norm_params.npz`

In [ ]:
# For Google Colab — download files
try:
    from google.colab import files
    files.download('premium_model_weights.pt')
    files.download('premium_norm_params.npz')
    print('Files downloaded!')
except:
    print('Not in Colab. Files saved locally.')
    print('Copy premium_model_weights.pt -> backend/ml/premium/model_weights.pt')
    print('Copy premium_norm_params.npz -> backend/ml/premium/norm_params.npz')